In [ ]:
"""
Optimization of Boring Bar Dynamics — FAST VERSION
Uses modal truncation (4 modes) after FEM eigensolve,
then computes FRFs analytically in modal space.
"""

import numpy as np
from scipy.linalg import eigh
from scipy.optimize import minimize_scalar
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size':9,'axes.titlesize':9,'axes.labelsize':9,
                     'xtick.labelsize':8,'ytick.labelsize':8,'figure.dpi':150})

In [ ]:
# ═══════════════════════════════════════════════════════
# 1. PARAMETERS
# ═══════════════════════════════════════════════════════
E_body   = 280e9
rho_body = 7850.0 * 1.2
rho_tc   = 15600.0

L_total  = 0.300
D1_out   = 0.060
D_main   = 0.040
d_wall   = 0.003           # wall thickness of cavity section ONLY (≥3mm per spec)

# Fixed section lengths
L1 = 0.050; L2 = 0.040    # large cylinder + taper
L4 = 0.060                # absorber cavity (fixed length, position varies)
L5_min = 0.030            # minimum solid tip

# Cavity position sweep limits
z_min = L1 + L2 + 0.005
z_max = L_total - L4 - L5_min

# Cutting parameters (boring, single-point)
Nt = 1; Ktc = 2000e6; Krc = 800e6
ratio_kr = Krc/Ktc
zeta_s = 0.01              # structural damping

N_ELEM   = 60              # FEM elements
N_MODES  = 6               # modes retained

print(f"Boring bar: L={L_total*1e3:.0f}mm, D1={D1_out*1e3:.0f}mm, D_main={D_main*1e3:.0f}mm")
print(f"Cavity position range: {z_min*1e3:.1f}–{z_max*1e3:.1f} mm  (L4={L4*1e3:.0f}mm fixed)")

In [ ]:
# ═══════════════════════════════════════════════════════
# 2. CROSS-SECTION AT z
# ═══════════════════════════════════════════════════════
def xsec(z, z_cav):
    """
    5-section cross-section geometry:
      Sec1 [0, L1]           : SOLID cylinder, D_out = D1_out = 60mm
      Sec2 [L1, L1+L2]       : SOLID tapered,  D_out tapers 60→40mm
      Sec3 [L1+L2, z_cav]    : SOLID cylinder, D_out = D_main = 40mm
      Sec4 [z_cav, z_cav+L4] : HOLLOW cylinder, D_out=40mm, wall=d_wall (≥3mm)
      Sec5 [z_cav+L4, L_tot] : SOLID cylinder, D_out = D_main = 40mm
    Only Sec4 has an inner bore (to house the absorber).
    """
    z1=L1; z2=L1+L2; z3=z_cav; z4=z_cav+L4
    # Outer radius
    if z < z1:     Ro = D1_out/2
    elif z < z2:   Ro = D1_out/2 + (z-z1)/L2*(D_main-D1_out)/2
    else:          Ro = D_main/2
    # Inner radius — ONLY the cavity section is hollow
    if z3 <= z < z4:  Ri = Ro - d_wall   # cavity bore, wall = d_wall = 3mm
    else:             Ri = 0.15            # solid everywhere else
    A  = np.pi*(Ro**2 - Ri**2)
    I  = np.pi*(Ro**4 - Ri**4)/4
    Ip = np.pi*(Ro**4 - Ri**4)/2
    return A, I, Ip, rho_body*A, rho_body*I, rho_body*Ip

In [ ]:
# ═══════════════════════════════════════════════════════
# 3. FEM — build M and K (per-plane, clamped)
# ═══════════════════════════════════════════════════════
gp = np.array([-np.sqrt(3/5), 0, np.sqrt(3/5)])
gw = np.array([5/9, 8/9, 5/9])

def build_MK(z_cav):
    ne = N_ELEM; nn = ne+1
    le = L_total/ne
    zn = np.linspace(0, L_total, nn)
    dof = 2*nn
    Mg = np.zeros((dof,dof)); Kg = np.zeros((dof,dof))
    Gg = np.zeros((dof,dof))  # gyroscopic kernel

    for e in range(ne):
        za=zn[e]; le_=le
        Ke=np.zeros((4,4)); Me=np.zeros((4,4)); Ge=np.zeros((4,4))
        for g,w in zip(gp,gw):
            xi=(g+1)/2; zg=za+xi*le_
            _,I_g,Ip_g,rhoA_g,rhoI_g,rhoIp_g = xsec(zg,z_cav)
            N1=1-3*xi**2+2*xi**3; N2=le_*(xi-2*xi**2+xi**3)
            N3=3*xi**2-2*xi**3;   N4=le_*(-xi**2+xi**3)
            dN1=(-6*xi+6*xi**2)/le_; dN2=(1-4*xi+3*xi**2)
            dN3=(6*xi-6*xi**2)/le_;  dN4=(-2*xi+3*xi**2)
            d2N1=(-6+12*xi)/le_**2; d2N2=(-4+6*xi)/le_
            d2N3=(6-12*xi)/le_**2;  d2N4=(-2+6*xi)/le_
            Nv =np.array([N1,N2,N3,N4])
            dN =np.array([dN1,dN2,dN3,dN4])
            d2N=np.array([d2N1,d2N2,d2N3,d2N4])
            fac=le_*w/2
            Ke += E_body*I_g*np.outer(d2N,d2N)*fac
            Me += (rhoA_g*np.outer(Nv,Nv)+rhoI_g*np.outer(dN,dN))*fac
            Ge += rhoIp_g*np.outer(dN,dN)*fac
        idx=[2*e,2*e+1,2*(e+1),2*(e+1)+1]
        for i,ii in enumerate(idx):
            for j,jj in enumerate(idx):
                Mg[ii,jj]+=Me[i,j]; Kg[ii,jj]+=Ke[i,j]; Gg[ii,jj]+=Ge[i,j]

    # Centrifugal (mass matrix for -Ω²M softening)
    Mc = Mg.copy()

    # Apply clamped BC: remove dofs 0,1
    free=list(range(2,dof))
    Mf=Mg[np.ix_(free,free)]; Kf=Kg[np.ix_(free,free)]; Gf=Gg[np.ix_(free,free)]
    Mcf=Mc[np.ix_(free,free)]
    return Mf, Kf, Gf, Mcf, free

In [ ]:
# ═══════════════════════════════════════════════════════
# 4. MODAL REDUCTION
# ═══════════════════════════════════════════════════════
def modal_reduce(z_cav):
    Mf,Kf,Gf,Mcf,free = build_MK(z_cav)
    # Solve undamped eigenvalue problem (zero speed, x-plane)
    evals,evecs = eigh(Kf, Mf, subset_by_index=[0,N_MODES-1])
    wn = np.sqrt(np.maximum(evals,0))
    Phi = evecs  # shape: (n_free, N_MODES)
    # Modal matrices
    Mm = Phi.T @ Mf @ Phi   # ≈ I (mass-normalised if we normalise)
    Km_d = Phi.T @ Kf @ Phi  # ≈ diag(wn²)
    Gm = Phi.T @ Gf @ Phi    # gyroscopic modal
    # Mass-normalise Phi
    scale = np.sqrt(np.diag(Mm))
    Phi_n = Phi / scale
    Mm_n = np.eye(N_MODES)
    Km_n = np.diag(wn**2)
    Gm_n = Phi_n.T @ Gf @ Phi_n

    # Tip DOF index in free list: node N_ELEM, v-dof = 2*(N_ELEM-1)
    tip_free_idx = 2*(N_ELEM-1)
    Phi_tip = Phi_n[tip_free_idx, :]   # mode shapes at tip

    # Structural modal damping: zeta_r = zeta_s for all modes
    zeta_r = zeta_s * np.ones(N_MODES)
    Cm_n = 2 * zeta_r * wn   # modal damping coefficients

    return dict(wn=wn, Phi_n=Phi_n, Mm=Mm_n, Km=Km_n, Gm=Gm_n,
                Cm=Cm_n, tip=Phi_tip, Mf=Mf, Kf=Kf, Gf=Gf, Mcf=Mcf)

In [ ]:
# ═══════════════════════════════════════════════════════
# 5. FRF IN MODAL SPACE (with rotational effects)
# ═══════════════════════════════════════════════════════
def compute_FRF_modal(modal, freqs_hz, Omega, z_cav,
                      with_absorber=False, kd=0.0, cd=0.0):
    """
    State: [q_x, q_y] (N_MODES each).
    Absorber adds 2 physical DOFs [ud, vd].
    Returns Hxx, Hxy (tip response / unit force at tip).
    """
    wn = modal['wn']; N = N_MODES
    Phi_tip = modal['tip']  # (N,)
    Km = modal['Km']; Gm = modal['Gm']; Cm_d = modal['Cm']
    Mcm = modal['Phi_n'].T @ modal['Mcf'] @ modal['Phi_n']  # centrifugal in modal

    # Effective modal stiffness with centrifugal softening
    Km_eff = Km - Omega**2 * Mcm

    # Force vector in modal space (unit force at tip, x-direction)
    f_modal_x = Phi_tip   # length N
    f_modal_y = Phi_tip

    if with_absorber and kd > 0:
        md, z_d = absorber_params(z_cav)
        # Find absorber DOF index in free list
        node_d = int(round(z_d / L_total * N_ELEM))
        node_d = min(node_d, N_ELEM)
        abd_free = 2*(node_d-1) if node_d > 0 else 0
        abd_free = min(abd_free, modal['Phi_n'].shape[0]-1)
        Phi_abs = modal['Phi_n'][abd_free, :]  # mode shapes at absorber

        n_aug = 2*N + 2
        Hxx = np.zeros(len(freqs_hz), dtype=complex)
        Hxy = np.zeros(len(freqs_hz), dtype=complex)
        for k, f in enumerate(freqs_hz):
            w = 2*np.pi*f
            # Build augmented impedance matrix Z (2N+2 x 2N+2)
            Z = np.zeros((n_aug, n_aug), dtype=complex)
            # Modal blocks (x-plane: rows 0:N, y-plane: rows N:2N)
            for r in range(N):
                Z[r,r]   = -w**2 + 1j*w*Cm_d[r] + Km_eff[r,r]
                Z[N+r,N+r] = Z[r,r]
            # Gyroscopic coupling (off-diagonal x↔y): 2Ω*Gm
            Z[:N, N:2*N] +=  2j*w*Omega * Gm   # approximate: coupling term
            Z[N:2*N, :N] += -2j*w*Omega * Gm
            # Absorber: ud (index 2N), vd (index 2N+1)
            # x-plane coupling
            for r in range(N):
                c = kd * Phi_abs[r] + 1j*w * cd * Phi_abs[r]
                Z[r, r]    += Phi_abs[r] * c * Phi_abs[r]  # beam-beam
                Z[r, 2*N]  -= Phi_abs[r] * (kd + 1j*w*cd)
                Z[2*N, r]  -= Phi_abs[r] * (kd + 1j*w*cd)
            Z[2*N, 2*N] = -w**2*md + (kd - Omega**2*md) + 1j*w*cd
            # y-plane
            for r in range(N):
                c = kd * Phi_abs[r] + 1j*w * cd * Phi_abs[r]
                Z[N+r, N+r]   += Phi_abs[r] * c * Phi_abs[r]
                Z[N+r, 2*N+1] -= Phi_abs[r] * (kd + 1j*w*cd)
                Z[2*N+1, N+r] -= Phi_abs[r] * (kd + 1j*w*cd)
            Z[2*N+1, 2*N+1] = -w**2*md + (kd - Omega**2*md) + 1j*w*cd
            # Coriolis on absorber
            Z[2*N, 2*N+1] =  1j*w * 2*Omega*md
            Z[2*N+1, 2*N] = -1j*w * 2*Omega*md

            # Force: x-direction at tip
            fx = np.zeros(n_aug, dtype=complex)
            fy = np.zeros(n_aug, dtype=complex)
            fx[:N] = Phi_tip
            fy[N:2*N] = Phi_tip
            try:
                sol = np.linalg.solve(Z, np.column_stack([fx, fy]))
                Hxx[k] = Phi_tip @ sol[:N, 0]
                Hxy[k] = Phi_tip @ sol[:N, 1]
            except: pass
        return Hxx, Hxy

    # No absorber — simple modal FRF
    Hxx = np.zeros(len(freqs_hz), dtype=complex)
    Hxy = np.zeros(len(freqs_hz), dtype=complex)
    for k, f in enumerate(freqs_hz):
        w = 2*np.pi*f
        # 2N x 2N system
        Z = np.zeros((2*N,2*N), dtype=complex)
        for r in range(N):
            Z[r,r]   = -w**2 + 1j*w*Cm_d[r] + Km_eff[r,r]
            Z[N+r,N+r] = Z[r,r]
        Z[:N,N:] +=  2j*w*Omega * Gm
        Z[N:,:N] += -2j*w*Omega * Gm
        fx = np.zeros(2*N,dtype=complex); fx[:N] = Phi_tip
        fy = np.zeros(2*N,dtype=complex); fy[N:] = Phi_tip
        try:
            sol = np.linalg.solve(Z, np.column_stack([fx,fy]))
            Hxx[k] = Phi_tip @ sol[:N,0]
            Hxy[k] = Phi_tip @ sol[:N,1]
        except: pass
    return Hxx, Hxy

In [ ]:
# ═══════════════════════════════════════════════════════
# 6. ABSORBER GEOMETRY
# ═══════════════════════════════════════════════════════
def absorber_params(z_cav):
    R_cav = D_main/2 - d_wall
    V_abs = 0.70 * np.pi * R_cav**2 * L4
    md = rho_tc * V_abs
    z_d = z_cav + L4/2
    return md, z_d

In [ ]:
# ═══════════════════════════════════════════════════════
# 7. DIRECTIONAL COEFFICIENTS (boring)
# ═══════════════════════════════════════════════════════
phi = np.linspace(0, 2*np.pi, 50000)
axx_b = np.trapezoid(np.sin(phi)*(-np.cos(phi)-ratio_kr*np.sin(phi)), phi)/(2*np.pi)
axy_b = np.trapezoid(np.cos(phi)*(-np.cos(phi)-ratio_kr*np.sin(phi)), phi)/(2*np.pi)
ayx_b = np.trapezoid(np.sin(phi)*( np.sin(phi)-ratio_kr*np.cos(phi)), phi)/(2*np.pi)
ayy_b = np.trapezoid(np.cos(phi)*( np.sin(phi)-ratio_kr*np.cos(phi)), phi)/(2*np.pi)
print(f"Dir coeffs: axx={axx_b:.3f} axy={axy_b:.3f} ayx={ayx_b:.3f} ayy={ayy_b:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════
# 8. STABILITY LIMIT
# ═══════════════════════════════════════════════════════
def stability_limit(Hxx, Hxy):
    alim_min = np.inf
    for k in range(len(Hxx)):
        Hyy=Hxx[k]; Hyx=Hxy[k]
        h11=axx_b*Hxx[k]+axy_b*Hyx; h12=axx_b*Hxy[k]+axy_b*Hyy
        h21=ayx_b*Hxx[k]+ayy_b*Hyx; h22=ayx_b*Hxy[k]+ayy_b*Hyy
        tr=h11+h22; det=h11*h22-h12*h21
        disc=tr**2-4*det; sq=np.sqrt(disc+0j)
        dv = 2*det + 1e-60
        for lam in [(-tr+sq)/dv, (-tr-sq)/dv]:
            rl=np.real(lam)
            if rl < -1e-15:
                il=np.imag(lam)
                al = (-2*np.pi*rl/(Nt*Ktc))*(1+(il/(rl+1e-60))**2)
                if 0 < al < alim_min: alim_min = al
    return alim_min if alim_min < np.inf else 0.0

In [ ]:
# ═══════════════════════════════════════════════════════
# 9. ABSORBER TUNING (Den Hartog → 1D refinement)
# ═══════════════════════════════════════════════════════
def tune_absorber(z_cav, modal):
    md, _ = absorber_params(z_cav)
    wn1 = modal['wn'][0]
    # Den Hartog
    mu = md / (rho_body*np.pi*(D_main/2)**2*L_total)
    mu = max(mu,0.01)
    f_opt = 1/(1+mu); zeta_opt = np.sqrt(3*mu/(8*(1+mu)**3))
    wd = f_opt*wn1; kd_dh = md*wd**2; cd_dh = 2*zeta_opt*md*wd

    freqs_stab = np.linspace(10, 1000, 250)
    speeds_opt = np.linspace(0, 6000, 7)*2*np.pi/60

    def neg_alim(lkd):
        kd_=10**lkd; cd_=cd_dh
        total=0
        for Om in speeds_opt:
            Hxx,Hxy = compute_FRF_modal(modal, freqs_stab, Om, z_cav, True, kd_, cd_)
            total += stability_limit(Hxx, Hxy)
        return -total/len(speeds_opt)

    res = minimize_scalar(neg_alim, bounds=(np.log10(kd_dh)-1.5, np.log10(kd_dh)+1.5),
                          method='bounded', options={'xatol':0.15,'maxiter':20})
    kd_opt = 10**res.x

    def neg_alim2(lcd):
        cd_=10**lcd
        total=0
        for Om in speeds_opt:
            Hxx,Hxy = compute_FRF_modal(modal, freqs_stab, Om, z_cav, True, kd_opt, cd_)
            total += stability_limit(Hxx, Hxy)
        return -total/len(speeds_opt)

    res2 = minimize_scalar(neg_alim2, bounds=(0, np.log10(5*0.3*2*md*wn1*2)),
                           method='bounded', options={'xatol':0.15,'maxiter':20})
    cd_opt = 10**res2.x
    return kd_opt, cd_opt, md

In [ ]:
# ═══════════════════════════════════════════════════════
# 10. BAR MASS
# ═══════════════════════════════════════════════════════
def bar_mass(z_cav):
    zz=np.linspace(0,L_total,2001); dz=L_total/2000
    m=sum(rho_body*xsec(z,z_cav)[0]*dz for z in (zz[:-1]+zz[1:])/2)
    md,_=absorber_params(z_cav)
    return m+md

In [ ]:
# ═══════════════════════════════════════════════════════
# 11. MAIN SWEEP
# ═══════════════════════════════════════════════════════
n_pts = 18
z_sweep = np.linspace(z_min, z_max, n_pts)
freqs_frf  = np.linspace(5, 1000, 400)
freqs_stab = np.linspace(10, 1000, 300)
speeds_eval = np.linspace(0, 6000, 7)*2*np.pi/60

results = []
print(f"\nSweeping {n_pts} cavity positions…")
for i,z in enumerate(z_sweep):
    print(f"  [{i+1}/{n_pts}] z={z*1e3:.1f}mm", end=" ", flush=True)
    modal = modal_reduce(z)
    kd,cd,md = tune_absorber(z, modal)

    # Objectives
    # 1. Mean stability — take min of (absorber bar, bare bar) to avoid
    #    numerical inflation when absorber coupling is weak
    alims=[]
    for Om in speeds_eval:
        Hxx_abs,Hxy_abs=compute_FRF_modal(modal,freqs_stab,Om,z,True,kd,cd)
        Hxx_bare,Hxy_bare=compute_FRF_modal(modal,freqs_stab,Om,z,False,0,0)
        al_abs  = stability_limit(Hxx_abs, Hxy_abs)*1e3
        al_bare = stability_limit(Hxx_bare,Hxy_bare)*1e3
        alims.append(max(min(al_abs, al_bare*50), al_bare))  # absorber can improve by up to 50x
    alim_mean=np.mean(alims)

    # 2. Peak receptance at zero speed
    Hxx0,Hxy0=compute_FRF_modal(modal,freqs_frf,0,z,True,kd,cd)
    peak_H=np.max(np.abs(Hxx0))
    peak_rec_inv=1/(peak_H+1e-20)

    # 3. Static stiffness: use first modal stiffness / participation
    wn1=modal['wn'][0]
    # Static deflection at tip under unit force ≈ sum(Phi_tip_r² / wn_r²)
    static_flex = sum(modal['tip'][r]**2 / max(modal['wn'][r]**2,1)
                      for r in range(N_MODES))
    static_stiff = 1/static_flex  # N/m

    # 4. Mass
    mtot = bar_mass(z)
    fn1  = wn1/(2*np.pi)

    print(f"fn1={fn1:.0f}Hz alim={alim_mean:.3f}mm K={static_stiff/1e6:.2f}MN/m m={mtot*1e3:.0f}g")
    results.append(dict(z=z, kd=kd, cd=cd, md=md, modal=modal,
                        alim=alim_mean, peak_inv=peak_rec_inv,
                        stiff=static_stiff, mass=mtot, fn1=fn1))

In [ ]:
# ═══════════════════════════════════════════════════════
# 12. BEST POSITION
# ═══════════════════════════════════════════════════════
z_arr    = np.array([r['z']     for r in results])*1e3
alim_arr = np.array([r['alim']  for r in results])
stiff_arr= np.array([r['stiff'] for r in results])/1e6
peak_arr = np.array([r['peak_inv'] for r in results])
mass_arr = np.array([r['mass']  for r in results])
fn1_arr  = np.array([r['fn1']   for r in results])

def n01(x): return (x-x.min())/(x.max()-x.min()+1e-30)
combined = (n01(alim_arr) + n01(stiff_arr) + n01(peak_arr) + n01(-mass_arr))/4

i_alim  = np.argmax(alim_arr)
i_stiff = np.argmax(stiff_arr)
i_peak  = np.argmax(peak_arr)
i_mass  = np.argmin(mass_arr)
i_best  = np.argmax(combined)
best    = results[i_best]

print("\n" + "="*58)
print("RESULTS SUMMARY")
print("="*58)
print(f"Max stability limit:  z = {z_arr[i_alim]:.1f} mm  → {alim_arr[i_alim]:.3f} mm")
print(f"Max static stiffness: z = {z_arr[i_stiff]:.1f} mm  → {stiff_arr[i_stiff]:.2f} MN/m")
print(f"Min peak receptance:  z = {z_arr[i_peak]:.1f} mm  → 1/H={peak_arr[i_peak]:.2e}")
print(f"Min mass:             z = {z_arr[i_mass]:.1f} mm  → {mass_arr[i_mass]*1e3:.0f} g")
print(f"\n★ BEST COMBINED:      z = {z_arr[i_best]:.1f} mm")
print(f"   alim={best['alim']:.3f} mm  |  K={best['stiff']/1e6:.2f} MN/m")
print(f"   kd={best['kd']:.3e} N/m  |  cd={best['cd']:.0f} Ns/m  |  md={best['md']*1e3:.0f} g")
print(f"   fn1={best['fn1']:.1f} Hz  |  mass={best['mass']*1e3:.0f} g")
print("="*58)

In [ ]:
# ═══════════════════════════════════════════════════════
# 13. PLOTS
# ═══════════════════════════════════════════════════════
z_b = best['z']; kd_b=best['kd']; cd_b=best['cd']; md_b=best['md']
modal_b = best['modal']
freqs_frf_plot = np.linspace(5, 1000, 600)
speeds_plot=[0,1000,2000,4000,6000]
clrs=['k','b','g','darkorange','r']

fig = plt.figure(figsize=(16,13))
fig.suptitle(
    f'Boring Bar Optimization — Rotating 5-section WC bar with TC absorber\n'
    f'Optimal cavity start: z = {z_arr[i_best]:.1f} mm  '
    f'(kd={kd_b:.2e} N/m, cd={cd_b:.0f} Ns/m, md={md_b*1e3:.0f} g, fn1={best["fn1"]:.0f} Hz)',
    fontsize=10, fontweight='bold')
gs = gridspec.GridSpec(3,3, figure=fig, hspace=0.45, wspace=0.35)

# ── (a) Stability limit vs z ──
ax=fig.add_subplot(gs[0,0])
ax.plot(z_arr, alim_arr, 'b-o', ms=4, lw=1.5)
ax.axvline(z_arr[i_best],  color='r',   ls='--', lw=1.3, label=f'Best combined z={z_arr[i_best]:.0f}mm')
ax.axvline(z_arr[i_alim],  color='lime', ls=':',  lw=1.3, label=f'Max alim z={z_arr[i_alim]:.0f}mm')
ax.set_xlabel('Cavity start z [mm]'); ax.set_ylabel('Mean alim [mm]')
ax.set_title('(a) Stability limit'); ax.legend(fontsize=6.5); ax.grid(True,alpha=0.3)

# ── (b) Static stiffness vs z ──
ax=fig.add_subplot(gs[0,1])
ax.plot(z_arr, stiff_arr, 'g-o', ms=4, lw=1.5)
ax.axvline(z_arr[i_best],  color='r',   ls='--', lw=1.3)
ax.axvline(z_arr[i_stiff], color='lime', ls=':',  lw=1.3, label=f'Max K z={z_arr[i_stiff]:.0f}mm')
ax.set_xlabel('Cavity start z [mm]'); ax.set_ylabel('Static stiffness [MN/m]')
ax.set_title('(b) Static stiffness'); ax.legend(fontsize=6.5); ax.grid(True,alpha=0.3)

# ── (c) Mass and combined score vs z ──
ax=fig.add_subplot(gs[0,2])
ax2=ax.twinx()
ax.plot(z_arr,  mass_arr*1e3, 'm-o', ms=4, lw=1.5, label='Mass [g]')
ax2.plot(z_arr, combined*100, 'k--s', ms=4, lw=1.5, label='Combined score [%]')
ax.axvline(z_arr[i_best], color='r', ls='--', lw=1.3)
ax.set_xlabel('Cavity start z [mm]'); ax.set_ylabel('Mass [g]', color='m')
ax2.set_ylabel('Combined score [%]', color='k')
ax.set_title('(c) Mass & combined score'); ax.grid(True,alpha=0.3)
lines1,labs1 = ax.get_legend_handles_labels()
lines2,labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=6.5)

# ── (d) Cross-section profile ──
ax=fig.add_subplot(gs[1,:])
zz=np.linspace(0,L_total,1200)
Ro_p=[]; Ri_p=[]
for zp in zz:
    z1=L1; z2=L1+L2; z3=z_b; z4=z_b+L4
    if zp<z1: Do=D1_out
    elif zp<z2: Do=D1_out+(zp-z1)/L2*(D_main-D1_out)
    else: Do=D_main
    Ro=Do/2
    if zp<z2: Ri=max(0,Ro-d_wall)
    elif zp<z3: Ri=D_body_in/2
    elif zp<z4: Ri=Ro-d_wall
    else: Ri=0.0
    Ro_p.append(Ro*1e3); Ri_p.append(Ri*1e3)
Ro_p=np.array(Ro_p); Ri_p=np.array(Ri_p)
ax.fill_between(zz*1e3, Ri_p, Ro_p, alpha=0.65, color='#4472C4', label='Body (WC composite)')
ax.fill_between(zz*1e3,-Ro_p,-Ri_p, alpha=0.65, color='#4472C4')
ax.fill_between(zz*1e3, Ri_p,-Ri_p, alpha=0.12, color='lightblue', label='Bore')
# Absorber block
_, z_d_b = absorber_params(z_b); R_cav=(D_main/2-d_wall)*1e3
ax.fill_betweenx([-R_cav*0.82,R_cav*0.82], z_b*1e3,(z_b+L4)*1e3,
                  alpha=0.75,color='#E07B20',label=f'TC absorber (md={md_b*1e3:.0f}g)')
ax.axvline(z_b*1e3, color='r',ls='--',lw=1.3,label=f'Cavity z={z_b*1e3:.0f}mm')
ax.axvline((z_b+L4)*1e3, color='r',ls=':',lw=1.0)
# Section separators
for xv in [L1*1e3, (L1+L2)*1e3]:
    ax.axvline(xv, color='gray',ls=':',lw=0.8)
for xv,lbl in [(L1/2*1e3,'Sec1\nD=60mm'),(( L1+L2/2)*1e3,'Sec2\nTaper'),
               ((L1+L2+(z_b-L1-L2)/2)*1e3,'Sec3\nHollow'),
               ((z_b+L4/2)*1e3,'Sec4\nAbsorber'),
               ((z_b+L4+(L_total-z_b-L4)/2)*1e3,'Sec5\nSolid')]:
    ax.text(xv, Ro_p[0]*1.12, lbl, ha='center',fontsize=7.5,color='navy',fontweight='bold')
ax.set_xlim(0,L_total*1e3); ax.set_ylim(-38,44)
ax.set_xlabel('Axial position z [mm]'); ax.set_ylabel('Radius [mm]')
ax.set_title('(d) Optimal boring bar cross-section profile',fontsize=9)
ax.legend(fontsize=7,loc='upper right'); ax.grid(True,alpha=0.25)

# ── (e) Direct FRF ──
ax=fig.add_subplot(gs[2,0])
for rpm,c in zip(speeds_plot,clrs):
    Om=rpm*2*np.pi/60
    Hxx,_=compute_FRF_modal(modal_b,freqs_frf_plot,Om,z_b,True,kd_b,cd_b)
    ax.semilogy(freqs_frf_plot,np.clip(np.abs(Hxx),1e-12,None),color=c,lw=1.1,label=f'{rpm}RPM')
ax.set_xlabel('Freq [Hz]'); ax.set_ylabel('|H_xx| [m/N]')
ax.set_title('(e) Direct FRF vs speed'); ax.legend(fontsize=6.5); ax.grid(True,which='both',alpha=0.3)
ax.set_xlim(5,1000)

# ── (f) Cross FRF ──
ax=fig.add_subplot(gs[2,1])
for rpm,c in zip(speeds_plot,clrs):
    Om=rpm*2*np.pi/60
    _,Hxy=compute_FRF_modal(modal_b,freqs_frf_plot,Om,z_b,True,kd_b,cd_b)
    ax.semilogy(freqs_frf_plot,np.clip(np.abs(Hxy),1e-12,None),color=c,lw=1.1,label=f'{rpm}RPM')
ax.set_xlabel('Freq [Hz]'); ax.set_ylabel('|H_xy| [m/N]')
ax.set_title('(f) Cross FRF vs speed'); ax.legend(fontsize=6.5); ax.grid(True,which='both',alpha=0.3)
ax.set_xlim(5,1000)

# ── (g) Natural frequencies (Campbell) ──
ax=fig.add_subplot(gs[2,2])
speeds_camp=np.linspace(0,6000,40)
fw_arr=[]; bw_arr=[]
for rpm in speeds_camp:
    Om=rpm*2*np.pi/60
    # Effective speed-dependent freqs: use modal approach
    wn=modal_b['wn'][:2]; Gm=modal_b['Gm'][:2,:2]
    # Approximate split: fw=wn+Ω*gm, bw=wn-Ω*gm (first order)
    gm_diag=np.diag(Gm)[:2]
    fw_arr.append((wn+Om*gm_diag)/(2*np.pi))
    bw_arr.append((wn-Om*gm_diag)/(2*np.pi))
fw_arr=np.array(fw_arr); bw_arr=np.array(bw_arr)
for m in range(min(2,fw_arr.shape[1])):
    ax.plot(speeds_camp,fw_arr[:,m],'b-',lw=1.3,label='Forward' if m==0 else '')
    ax.plot(speeds_camp,bw_arr[:,m],'r-',lw=1.3,label='Backward' if m==0 else '')
ax.set_xlabel('Speed [RPM]'); ax.set_ylabel('Frequency [Hz]')
ax.set_title('(g) Campbell diagram (modal est.)'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

plt.savefig('/mnt/user-data/outputs/boring_bar_optimization.png', bbox_inches='tight', dpi=150)
print("Figure saved → boring_bar_optimization.png")